# Stream Host — Notebook Spike Fase 0 (VoiceStudio backend saja)

Notebook ini adalah **versi spike produk Stream Host** — semata untuk
membuktikan rantai engine produk (gerbang **S1/S2/S3**, dokumen 04),
bukan notebook final yang dibagikan ke pembeli. Ia menjalankan backend
VoiceStudio **headless (tanpa UI bawaan)** di GPU T4 milik sesi Google Anda,
lalu membukanya lewat tunnel Cloudflare agar halaman uji/dashboard bisa
memanggilnya lintas-origin dengan **Nomor Mesin**.

## Berbeda dari notebook resmi VoiceStudio

- Frontend **tidak dibangun** dan tidak disajikan (UI satu-satunya = dashboard Stream Host).
- Model yang diunduh hanya **VoxCPM2 + Whisper**; **OmniVoice TIDAK diunduh** (bobotnya CC-BY-NC, dilarang untuk produk jualan — lihat NOTICE di akhir).
- Engine untuk generate **dipaksa `voxcpm2`** lewat `OMNIVOICE_TTS_BACKEND`.
- Backend dibuka lewat **cloudflared quick tunnel**; token pairing adalah kredensial API sesi (`OMNIVOICE_API_KEY`, header `Authorization: Bearer`, **bukan** parameter query).

## Cara menjalankan sesi spike

1. Runtime → Ubah jenis runtime → GPU **T4**.
2. Isi sel konfigurasi sesi **sebelum** menjalankan backend — perubahan env backend hanya berlaku saat proses backend dimulai.
3. Jalankan sel dari atas ke bawah. Tiap tahap memunculkan bukti hasilnya; jangan loncat tahap.

> **Catatan spike:** `OMNIVOICE_DATA_DIR` memakai `/content/omnivoice_data` (bawaan) dan `HF_HOME` default — *tanpa Google Drive* agar sesi pertama sederhana. Keduanya diganti pada notebook produk sesudah gerbang lulus.


## Bagian 1 — Persiapan sesi (GPU, clone & install backend)

In [ ]:
# ── Tahap 1. Cek GPU (T4) ────────────────────────────────────────────
# Confirms the runtime has an NVIDIA GPU. Everything still works on CPU, but
# a single sentence can take minutes instead of seconds.
import shutil
import subprocess

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
    print("GPU detected — you're good to go.")
else:
    print("=" * 74)
    print("WARNING: no NVIDIA GPU in this runtime.")
    print("Fix: menu bar -> Runtime -> Change runtime type -> T4 GPU,")
    print("then re-run this notebook from the top.")
    print("(Continuing anyway works, but generation will be very slow on CPU.)")
    print("=" * 74)


## Tahap 2 — Clone + install backend saja (tanpa build UI)

Sel ini clone repo bila belum ada, menginstal paket sistem,
lalu menginstal **backend saja** ke Python sistem Colab. Tahap build
frontend upstream (`bun install` + `build:web`) **dilewati seluruhnya**.

> **Jebakan yang dihindari di sini:** pemeriksaan import upstream memeriksa
> `from omnivoice.models.omnivoice import OmniVoice` — kelas engine OmniVoice di
> pustaka `omnivoice` milik VoiceStudio (bukan bobot modelnya, yang TIDAK
> diunduh sesi ini). Pemeriksaan ini **tidak diganti** demi idempotensi
> terhadap sel upstream; arti "OmniVoice tidak diunduh" adalah bobot
> `k2-fsa/OmniVoice` dari Hugging Face. Pencatatan ini apa adanya agar
> tidak disangka pelanggaran terhadap aturan spike.


In [ ]:
# ── Tahap 2. Clone + install backend saja (headless; tanpa build UI) ─────────
# Berbeda dari notebook resmi VoiceStudio, sel ini melewati build frontend
# (bun + build:web, ±2 menit) SELURUHNYA: tidak ada UI bawaan yang dibangun
# atau disajikan di sesi spike. Dashboard Stream Host adalah satu-satunya UI.
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/expremature/VoiceStudio"
REPO_DIR = "/content/VoiceStudio"
# Generous network timeouts (verbatim upstream wording kept):
os.environ.setdefault("UV_HTTP_TIMEOUT", "300")

def run(cmd, *, cwd=None, what=""):
    """Stream a command's output; stop the cell with an actionable message on failure."""
    shown = cmd if isinstance(cmd, str) else " ".join(cmd)
    print(f"\n$ {shown}")
    rc = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str)).returncode
    if rc != 0:
        raise SystemExit(
            f"\nFAILED: {what or shown} (exit code {rc}).\n"
            "Scroll up for the underlying error. Re-running this cell is safe.\n"
            "Still stuck? Open an issue with the output above:\n"
            "  https://github.com/expremature/VoiceStudio/issues\n"
            "  (fork Stream Host; upstream: https://github.com/debpalash/VoiceStudio/issues)"
        )

# 2a. Clone the repo (reused if already present)
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"Repo already present at {REPO_DIR} — updating it.")
    run(["git", "-C", REPO_DIR, "pull", "--ff-only"],
        what=f"git pull in {REPO_DIR} (if you changed files there, delete the folder and re-run this cell)")
else:
    run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], what="git clone")

# 2b. System packages (ffmpeg: audio/video processing; libsndfile1: soundfile)
run("apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1", what="apt-get install")

# 2c. (frontend build SENGAJA TIDAK ADA di sini — lihat catatan di atas.)

# 2d. Backend dependencies — exact same command the official Docker image uses.
if not shutil.which("uv"):
    run([sys.executable, "-m", "pip", "install", "-q", "uv"], what="pip install uv")
run(["uv", "pip", "install", "--system", "--no-cache",
     "--constraint", "deploy/torch-constraints.txt",
     "--override", "deploy/uv-overrides.txt", "."],
    cwd=REPO_DIR, what="backend install (uv pip install --system .)")

# 2e. cuDNN 8 side-install for CTranslate2 (WhisperX ASR on GPU). Non-fatal:
# without it, transcription falls back to the torch-native Whisper backend.
os.makedirs(os.path.join(REPO_DIR, ".venv"), exist_ok=True)
run([sys.executable, os.path.join(REPO_DIR, "scripts", "setup.py")], what="cuDNN 8 compat setup")

# 2f. VoxCPM2 engine package — PINNED ( upstream floor is >=2.0.3; pin so the
# spike is reproducible ). Path (a) in-process is primary for this spike:
# OMNIVOICE_TTS_BACKEND=voxcpm2 resolves to services.tts_backend.VoxCPM2Backend
# in this venv, which does `import voxcpm`. The one-click sidecar venv
# (engines/voxcpm2_subprocess) is only preferred once its own venv exists
# (see _effective_backend_class); the notebook does NOT create it.
run([sys.executable, "-m", "pip", "install", "-q", "voxcpm==2.0.3"],
    what="pip install voxcpm==2.0.3 (VoxCPM2 engine package, path (a))")

# 2g. Sanity check in a fresh interpreter (this kernel may hold a stale torch).
# Unchanged from upstream apart from the report line: upstream imports
# `from omnivoice.models.omnivoice import OmniVoice` — that is VoiceStudio's
# `omnivoice` *code package* class, NOT the k2-fsa/OmniVoice *model weights*
# (the weights are what this notebook never downloads).
run([sys.executable, "-c",
     "import torch, torchaudio, uvicorn, fastapi, transformers; "
     "print(f'torch {torch.__version__}, torchaudio {torchaudio.__version__}, "
     "transformers {transformers.__version__}, "
     "CUDA available: {torch.cuda.is_available()}'); "
     "from omnivoice.models.omnivoice import OmniVoice; "
     "from transformers import HiggsAudioV2TokenizerModel; "
     "import voxcpm; "
     "print(f'voxcpm package: {getattr(voxcpm, \"__version__\", \"unknown\")}'); "
     "print('Install OK - backend model stack imports cleanly')"],
    cwd=REPO_DIR, what="import sanity check")

print("Backend + voxcpm==2.0.3 installed. No frontend was built.")

In [ ]:
# ── Tahap 3. (Opsional) Hugging Face token ───────────────────────────────
# Only needed for GATED models — e.g. pyannote speaker diarization, used by
# video dubbing for multi-speaker detection. TTS, voice cloning, and voice
# design all work WITHOUT a token, so feel free to skip this cell.
#
# To use one: accept the model terms at
#   https://huggingface.co/pyannote/speaker-diarization-3.1
# then add a Colab Secret named HF_TOKEN (key icon in the left sidebar),
# toggle "Notebook access" ON, and re-run this cell.
import os

try:
    from google.colab import userdata
    _token = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = _token
    print("HF_TOKEN loaded from Colab Secrets — gated models (diarization) enabled.")
except Exception:
    print("No HF_TOKEN Colab Secret found — skipping. (Everything except gated "
          "diarization models works without it.)")


## Sel konfigurasi sesi — WAIB diisi sebelum Tahap 6

Sel berikutnya membangkitkan token pairing sesi dan menulis `env` untuk
proses backend. **Tiga aturan penulisan dari pemilik yang tidak boleh dilanggar:**

1. Token pairing (32 byte acak, hex) dibangkitkan dan `OMNIVOICE_API_KEY` **diekspor sebelum** `uvicorn` dijalankan — backend membaca env hanya saat start proses.
2. `OMNIVOICE_TTS_BACKEND=voxcpm2` diekspor bersamaan di sel yang sama.
3. Token **hanya** dikirim ke backend lewat header `Authorization: Bearer` — dilarang lewat parameter query apa pun (`?api_key=` bocor ke log & riwayat browser).

Origin halaman uji sudah ditetapkan **sebelum** notebook ini dijalankan:

| Item | Nilai |
|---|---|
| `OMNIVOICE_ALLOWED_ORIGINS` | `https://app.streamhost.id` (origin dashboard Stream Host untuk spike ini) |
| Runtime Colab | GPU **T4** |
| `OMNIVOICE_DATA_DIR` | `/content/omnivoice_data` (bawaan, efemeral) |
| `HF_HOME` | *default* (bawaan) — dicatat apakah installer menghormatinya |

> **Peringatan jujur soal 403:** Origin yang terdaftar adalah `https://app.streamhost.id`.
> Bila halaman uji T-02 Anda sajikan dari origin **lain** (mis. jupyter/codesandbox/localhost),
> semua panggilan dari halaman itu ditolak gerbang origin `403` — itu bukan kegagalan
> rantai, tapi ketidaksesuaian konfigurasi (dokumen 04 Butir 2.1). Perbaikannya:
> ubah nilai `SH_ALLOWED_ORIGIN` di sel konfigurasi menjadi origin halaman uji Anda
> persis (skema + host + port), lalu **restart backend** (hentikan proses di port 3900,
> jalankan ulang Tahap 6) — dan catat nilai yang Anda pakai di lembar S1.


In [ ]:
# ── Sel konfigurasi sesi (WAJIB sebelum Tahap 6) ────────────────────────────
# Tiga aturan pemilik (terkonfirmasi dari source backend, docs/api-auth.md):
#   1) Token pairing (32 byte acak, hex) dibangkitkan dan OMNIVOICE_API_KEY
#      diekspor SEBELUM uvicorn start — backend hanya membaca env saat proses.
#   2) Pemaksaan engine lewat OMNIVOICE_TTS_BACKEND=voxcpm2 (env > pilihan UI
#      tersimpan, default upstream "omnivoice").
#   3) Token hanya lewat header "Authorization: Bearer <token>"; DILARANG
#      parameter query ?api_key= (bocor ke log dan riwayat browser).
import os
import secrets

# ---- Origin halaman uji (tetapkan SEBELUM backend jalan, dokumen 04 §2.1)
# Default: dashboard Stream Host. Ganti nilai ini bila halaman uji spike
# Anda tidak disajikan dari https://app.streamhost.id, lalu restart backend
# (Tahap 6) dan catat nilai persis yang dipakai di lembar S1.
SH_ALLOWED_ORIGIN = "https://app.streamhost.id"

# ---- Token pairing sesi (hanya di memori kernel ini + di Nomor Mesin)
SH_PAIRING_TOKEN = secrets.token_hex(32)  # 32 byte acak → 64 karakter hex
os.environ["OMNIVOICE_API_KEY"] = SH_PAIRING_TOKEN
os.environ["OMNIVOICE_TTS_BACKEND"] = "voxcpm2"
os.environ["OMNIVOICE_SERVER_MODE"] = "1"
os.environ["OMNIVOICE_DATA_DIR"] = os.environ.get("OMNIVOICE_DATA_DIR", "/content/omnivoice_data")
os.environ.setdefault("HF_HOME", os.path.expanduser("~/.cache/huggingface"))

print("Sel konfigurasi sesi siap.")
print("  OMNIVOICE_TTS_BACKEND =", os.environ["OMNIVOICE_TTS_BACKEND"])
print("  OMNIVOICE_SERVER_MODE =", os.environ["OMNIVOICE_SERVER_MODE"])
print("  OMNIVOICE_DATA_DIR    =", os.environ["OMNIVOICE_DATA_DIR"])
print("  HF_HOME               =", os.environ["HF_HOME"])
print("  Origin terdaftar       =", SH_ALLOWED_ORIGIN)
print("  OMNIVOICE_API_KEY      = <di-set — nilai penuh tidak dicetak di sini>")
print("  (Token pairing tampil utuh hanya di Nomor Mesin pada sel Tahap 7.)")

assert os.environ.get("OMNIVOICE_API_KEY"), "Token pairing kosong — jangan lanjut."
assert os.environ.get("OMNIVOICE_TTS_BACKEND") == "voxcpm2", "Pemaksaan engine VoxCPM2 gagal diatur."

## Bagian 2 — Menjalankan engine (backend headless, unduh model, tunnel)

Urutan di bagian ini disengaja: **konfigurasi → backend (Tahap 6) → unduh model (Tahap 4) → tunnel + Nomor Mesin (Tahap 7)**.
Tahap 4 memakai endpoint install backend, jadi backend harus sudah jalan sebelum model bisa diunduh lewat installer. Nomor tahap mengikuti penomoran arsitektur Bagian 7 agar rujukan silang tetap persis.


In [ ]:
# ── Tahap 6. Jalankan backend headless (env sudah dikunci sel konfigurasi) ────
# uvicorn FastAPI di port 3900; env pairing dipasang SEBELUM proses start
# (aturan 1): OMNIVOICE_API_KEY + OMNIVOICE_TTS_BACKEND=voxcpm2 + SERVER_MODE.
# Backend sengaja di-bind ke 127.0.0.1; cloudflared (Tahap 7) adalah
# satu-satunya jalur masuk dari luar Colab.
#
# Allowed origins/hosts memakai mekanisme upstream untuk proxy Colab
# (notebook resmi sel 5 + core/browser_guard.py):
#   - OMNIVOICE_ALLOWED_ORIGINS menerima origin halaman uji (SH_ALLOWED_ORIGIN)
#   - OMNIVOICE_ALLOWED_HOSTS menerima suffix ".trycloudflare.com" persis
#     seperti daftar suffix yang diizinkan; "*" (wildcard host) tidak dipakai.
import json
import os
import subprocess
import sys
import time
import urllib.request

REPO_DIR = "/content/VoiceStudio"
PORT = 3900
LOG_PATH = "/content/omnivoice_backend.log"
HEALTH_URL = f"http://127.0.0.1:{PORT}/health"

# Bila sel konfigurasi sesi belum dijalankan, hentikan dengan pesan jelas
# (backend tanpa OMNIVOICE_API_KEY = tanpa autentikasi → kegagalan sesi).
for kunci in ("OMNIVOICE_API_KEY", "OMNIVOICE_TTS_BACKEND", "OMNIVOICE_SERVER_MODE"):
    if not os.environ.get(kunci):
        raise SystemExit(
            f"Tahap 6 DITOLAK: env {kunci} belum diatur.\n"
            "Jalankan 'Sel konfigurasi sesi' di atas lebih dulu "
            "(token pairing wajib di-ekspor SEBELUM backend jalan — aturan 1)."
        )

_CLOUDFLARE_SUFFIX = ".trycloudflare.com"

def _allowed_hosts():
    saat_ini = os.environ.get("OMNIVOICE_ALLOWED_HOSTS", "")
    nilai = [v.strip() for v in (saat_ini + f",{_CLOUDFLARE_SUFFIX}").split(",") if v.strip()]
    return ",".join(dict.fromkeys(nilai))

def health():
    try:
        with urllib.request.urlopen(HEALTH_URL, timeout=5) as r:
            return json.load(r)
    except Exception:
        return None

info = health()
if info and info.get("status") == "ok":
    print(f"Backend sudah jalan — {info}")
    print("CATATAN: backend yang sudah hidup TIDAK mengambil env sel-konfigurasi baru.")
    print("Bila Anda baru mengubah SH_ALLOWED_ORIGIN atau token, hentikan backend dulu:")
    print("  !kill $(lsof -t -i:3900)  lalu jalankan ulang Tahap 6.")
else:
    subprocess.run(f"kill -9 $(lsof -t -i:{PORT}) 2>/dev/null || true", shell=True)
    time.sleep(1)

    env = os.environ.copy()
    env["OMNIVOICE_SERVER_MODE"] = "1"
    env.setdefault("OMNIVOICE_DATA_DIR", "/content/omnivoice_data")
    env["PYTHONUNBUFFERED"] = "1"
    env["OMNIVOICE_ALLOWED_ORIGINS"] = SH_ALLOWED_ORIGIN
    env["OMNIVOICE_ALLOWED_HOSTS"] = _allowed_hosts()
    print("Env yang diteruskan ke backend:")
    print("  OMNIVOICE_ALLOWED_ORIGINS =", env["OMNIVOICE_ALLOWED_ORIGINS"])
    print("  OMNIVOICE_ALLOWED_HOSTS   =", env["OMNIVOICE_ALLOWED_HOSTS"])
    print("  OMNIVOICE_TTS_BACKEND     =", env["OMNIVOICE_TTS_BACKEND"])
    print("  OMNIVOICE_API_KEY          = <di-set, tidak dicetak>")

    log = open(LOG_PATH, "ab")
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "main:app",
         "--app-dir", "backend", "--host", "127.0.0.1", "--port", str(PORT)],
        cwd=REPO_DIR, env=env, stdout=log, stderr=subprocess.STDOUT,
    )
    print(f"Backend starting (PID {proc.pid}); log: {LOG_PATH}")

    deadline = time.time() + 300
    while time.time() < deadline:
        if proc.poll() is not None:
            break
        info = health()
        if info and info.get("status") == "ok":
            break
        print(".", end="", flush=True)
        time.sleep(3)
    print()

    if info and info.get("status") == "ok":
        print(f"Backend is up — {info}")
        dev = str(info.get("device", "")).lower()
        if "cuda" not in dev:
            print("PERINGATAN: device bukan CUDA — generate akan SANGAT lambat.")
            print("Pastikan runtime GPU T4 (Runtime → Ubah jenis runtime → T4), lalu jalankan ulang dari Tahap 1 (sesi ini bukan bukti S1/S2).")
        else:
            print("Device CUDA terdeteksi. Lanjut Tahap 4 (unduh model) lalu Tahap 7.")
    else:
        try:
            with open(LOG_PATH, "r", errors="replace") as f:
                tail = "".join(f.readlines()[-40:])
        except OSError:
            tail = "(no log file found)"
        raise SystemExit(
            "Backend did not become healthy within 5 minutes.\n"
            f"--- last lines of {LOG_PATH} ---\n{tail}\n--- end of log ---\n"
            "Re-running this cell is safe (Tahap 2 sudah terpasang)."
        )

## Tahap 4 — Unduh model di muka lewat endpoint install backend (VoxCPM2 + Whisper)

Backend belum dijalankan saat tahap ini di notebook resmi, jadi notebook ini
**memajukan urutan**: Tahap 4 memakai `POST http://127.0.0.1:3900/models/install`
dari sel loopback (sesudah backend jalan di Tahap 6, sebelum tunnel dibuka di Tahap 7).
Panggilan ini **tidak** memakai tunnel dan tidak memerlukan header dari halaman web.

Model yang diminta adalah dua katalog `backend/config/models.yaml` persis:
`openbmb/VoxCPM2` (±5 GB) dan `Systran/faster-whisper-large-v3` (±2,9 GB).
**OmniVoice (`k2-fsa/OmniVoice`) TIDAK pernah diminta, TIDAK pernah diunduh.**

Sel ini mencatat apakah installer menghormati `HF_HOME` (temuan S1).
Bila endpoint install gagal, fallback ke unduhan langsung `snapshot_download`
(hanya dipakai bila backend-nya sendiri tidak bisa) dicatat sebagai temuan,
bukan dijalankan diam-diam.


In [ ]:
# ── Tahap 4. Unduh model di muka lewat POST /models/install (loopback) ───────
# Dieksekusi SESUDAH backend jalan (Tahap 6) dan SEBELUM tunnel dibuka (Tahap 7).
# Sesuai keputusan pemilik (verifikasi saat S1 dari source backend):
#   - POST /models/install diminta satu kali per repo katalog
#     ("openbmb/VoxCPM2", "Systran/faster-whisper-large-v3")
#   - Polling GET /models/install/status sampai keduanya selesai (bukan aktif)
#   - Endpoint ini di-loopback 127.0.0.1; token pairing tidak terlibat di sini.
# OmniVoice (k2-fsa/OmniVoice) TIDAK pernah diminta di sini.
import json
import os
import sys
import time
import urllib.error
import urllib.request

BASE = "http://127.0.0.1:3900"
REPOS_WAJIB = ["openbmb/VoxCPM2", "Systran/faster-whisper-large-v3"]

def http_json(method, path, payload=None, timeout=1800):
    url = f"{BASE}{path}"
    data = json.dumps(payload).encode() if payload is not None else None
    headers = {"Content-Type": "application/json"}
    req = urllib.request.Request(url, data=data, headers=headers, method=method)
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.load(r)

print("HF_HOME resolved:", os.environ.get("HF_HOME", "(default ~/.cache/huggingface)"))
print("HF_HUB_CACHE:", os.environ.get("HF_HUB_CACHE", "(unset)"))
print("OMNIVOICE_TTS_BACKEND =", os.environ.get("OMNIVOICE_TTS_BACKEND"))
print(f"Tahap 4 dimulai: meminta POST /models/install untuk {REPOS_WAJIB}")

for repo in REPOS_WAJIB:
    print(f"\n→ POST /models/install repo_id={repo}")
    try:
        r = http_json("POST", "/models/install", {"repo_id": repo})
        print(f"   diterima: {r}")
    except urllib.error.HTTPError as e:
        body = e.read().decode("utf-8", errors="replace")
        if e.code == 400 and "Unknown model" in body:
            raise SystemExit(
                f"Tahap 4 GAGAL: repo {repo!r} tidak dikenal katalog backend.\n"
                f"Periksa models.yaml di clone ini (harus memuat repo_id persis).\n{body}"
            )
        raise SystemExit(f"Tahap 4 GAGAL pada POST /models/install {repo}: HTTP {e.code} {body[:800]}")

# Poll /models/install/status sampai keduanya keluar dari set aktif.
print("\nMenunggu install tuntas (poll /models/install/status tiap 15 dtk) ...")
deadline = time.time() + 2.5 * 3600  # generous: ±8 GB cold download di Colab
while time.time() < deadline:
    status = http_json("GET", "/models/install/status", timeout=120)
    jobs = {j["repo_id"]: j for j in status.get("jobs", [])}
    aktif = [j for r, j in jobs.items() if r in REPOS_WAJIB and j.get("state") in ("downloading", "cancelling")]
    gagal = [j for r, j in jobs.items() if r in REPOS_WAJIB and j.get("state") == "failed"]
    if aktif:
        for j in aktif:
            print(f"   ... {j['repo_id']}: {j.get('state')} downloaded={j.get('downloaded', '?')} / {j.get('total', '?')}")
    if gagal:
        raise SystemExit(
            "Tahap 4 GAGAL: install berakhir gagal untuk "
            + ", ".join(j["repo_id"] for j in gagal)
            + "\nDetail: " + json.dumps(gagal, indent=2)[:2000]
            + "\nTindakan: tunnel tidak dibuka. Catat detail ini di lembar"
            + " S1 (bila endpoint install backend gugur, keputusan fallback"
            + " snapshot_download wajib tercatat terpisah — jangan diam-diam)."
        )
    if not aktif:
        print("Kedua install selesai (tidak ada job aktif).")
        break
    time.sleep(15)
else:
    raise SystemExit("Tahap 4 timeout (2,5 jam) — timeout bukan lulus; catat dan lanjutkan sesi kedua dengan model ter-cache.")

# Verifikasi status terpasang lewat katalog backend (GET /models).
try:
    katalog = http_json("GET", "/models", timeout=120)
    print("\nVerifikasi katalog /models (terpasang?):")
    items = katalog.get("models", katalog if isinstance(katalog, list) else [])
    for item in items:
        rid = item.get("repo_id") or item.get("id")
        if rid in REPOS_WAJIB:
            print(f"   - {rid}: installed={item.get('installed', item.get('is_installed', '?'))}")
except Exception as e:
    print(f"   (verifikasi katalog /models tidak terbaca persis: {e}; hasil /models/install/status di atas adalah bukti terikat)")

print("\nTahap 4 SELESAI — status terpasang VoxCPM2 + Whisper terverifikasi sebelum tunnel dibuka.")

In [ ]:
# ── Tahap 7. Tunnel cloudflared + Nomor Mesin (satu sel tidak terpisah) ──────
# Nomor Mesin hanya tercetak sesudah tunnel hidup DAN /health lewat tunnel
# terbukti lolos dengan token pairing dari sel konfigurasi.
#
# Alur sel ini:
#   7a. Pasang cloudflared (bila belum ada).
#   7b. Jalankan quick tunnel --url http://127.0.0.1:3900; tangkap URL acaknya.
#   7c. Uji GET {url}/health DENGAN header Authorization: Bearer <token>
#       (bukti sebelum diumumkan). Tanpa token juga dicatat apa adanya.
#   7d. Susun + cetak Nomor Mesin persis format Arsitektur Bagian 8 dan 04 §2.2.
import base64
import hashlib
import json
import os
import subprocess
import time
import urllib.error
import urllib.request

SH_LOG_TUNNEL = "/content/cloudflared.log"
SH_SESSION_EXP_SECONDS = 12 * 3600  # perkiraan sesi berakhir (unix time)

# --- 7a. Pasang cloudflared bila belum ada ------------------------------------
if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run(
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        " -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
        shell=True, check=True,
    )
    print("cloudflared terpasang.")
else:
    print("cloudflared sudah ada — dipakai ulang.")

# --- 7b. Jalankan quick tunnel ------------------------------------------------
subprocess.run("pkill -f 'cloudflared tunnel' 2>/dev/null || true", shell=True)
time.sleep(1)
open(SH_LOG_TUNNEL, "w").close()
subprocess.Popen(
    "nohup cloudflared tunnel --url http://127.0.0.1:3900 --no-autoupdate"
    f" > {SH_LOG_TUNNEL} 2>&1 &",
    shell=True,
)
tunnel_url = None
deadline = time.time() + 120
while time.time() < deadline and not tunnel_url:
    try:
        with open(SH_LOG_TUNNEL, errors="replace") as f:
            isi = f.read()
    except OSError:
        isi = ""
    for kata in isi.split():
        bersih = kata.strip('",')
        if bersih.startswith("https://") and bersih.endswith(".trycloudflare.com"):
            tunnel_url = bersih
            break
    if not tunnel_url:
        time.sleep(3)
if not tunnel_url:
    try:
        with open(SH_LOG_TUNNEL, errors="replace") as f:
            print(f.read()[-3000:])
    except OSError:
        pass
    raise SystemExit(
        "Tahap 7 GAGAL: URL *.trycloudflare.com tidak muncul dalam 120 detik.\n"
        "Tunnel tidak dibuka, Nomor Mesin tidak diterbitkan; gauge S1 belum lulus."
    )
print(f"Tunnel hidup: {tunnel_url}")

# --- 7c. Bukti kesehatan lewat tunnel (DENGAN token, lalu TANPA token) --------
def _get_health(url, headers=None, timeout=30):
    req = urllib.request.Request(f"{url}/health", headers=headers or {}, method="GET")
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.status, json.load(r)
    except urllib.error.HTTPError as e:
        body = e.read().decode("utf-8", errors="replace")[:500]
        return e.code, {"detail": body}
    except Exception as e:
        return None, {"detail": f"{type(e).__name__}: {e}"}

token = os.environ.get("OMNIVOICE_API_KEY", "")
assert token, "Token pairing hilang dari env kernel — jalankan ulang sel konfigurasi, lalu restart backend (Tahap 6)."
hdr = {"Authorization": f"Bearer {token}"}
status_ok, data_ok = None, {}
for percobaan in range(20):
    status_ok, data_ok = _get_health(tunnel_url, headers=hdr)
    if status_ok == 200 and isinstance(data_ok, dict) and data_ok.get("status") == "ok":
        break
    time.sleep(3)
print(f"GET {{tunnel}}/health dengan Bearer -> {status_ok} {data_ok}")
status_tanpa, data_tanpa = _get_health(tunnel_url, headers={})
print(f"GET {{tunnel}}/health TANPA token -> {status_tanpa} {data_tanpa}")
if not (status_ok == 200 and isinstance(data_ok, dict) and data_ok.get("status") == "ok"):
    raise SystemExit(
        "Tahap 7 GAGAL: /health lewat tunnel dengan token pairing tidak sehat.\n"
        "Periksa: (1) backend masih hidup di 127.0.0.1:3900, (2) token dari sel "
        "konfigurasi sama dengan yang diekspor sebelum backend start, "
        "(3) OMNIVOICE_ALLOWED_HOSTS memuat suffix tunnel (Tahap 6)."
    )
device = str(data_ok.get("device", ""))
engine_version = str(data_ok.get("version", "unknown"))
print(f"Device: {device} | engine_version: {engine_version}")
if "cuda" not in device.lower():
    print("PERINGATAN: device bukan CUDA — generate akan SANGAT lambat (runtime T4 belum aktif).")

# --- 7d. Susun + CETAK Nomor Mesin (Arsitektur Bagian 8 / dokumen 04 §2.2) ---
# Nomor Mesin = "VS1-" + base64url(JSON) + "-" + checksum4
# JSON = {"v":1, "url": <tunnel>, "key": <token pairing>, "exp": <unix time>}
payload_obj = {
    "v": 1,
    "url": tunnel_url,
    "key": token,
    "exp": int(time.time()) + SH_SESSION_EXP_SECONDS,
}
payload_bytes = json.dumps(payload_obj, separators=(",", ":"), ensure_ascii=False).encode("utf-8")
payload_b64 = base64.urlsafe_b64encode(payload_bytes).decode("ascii")
checksum4 = hashlib.sha256(payload_bytes).hexdigest()[:4]
nomor_mesin = f"VS1-{payload_b64}-{checksum4}"

sep = "=" * 74
print("\n" + sep)
print("ENGINE SIAP — salin SELURUH Nomor Mesin di bawah ini ke halaman uji / dashboard:")
print(sep)
print(nomor_mesin)
print(sep)
print(f"URL tunnel   : {tunnel_url}")
print(f"Device       : {device}")
print("Status       : Terhubung (health check lolos dengan token pairing)")
print("Catatan      : Nomor Mesin ini RAHASIA sesi ini dan hanya berlaku selama sesi Colab ini hidup.")
print("               Jangan membagikannya ke publik; kerugian bila bocor terbatas pada kuota Colab sesi ini.")
print(sep)

In [ ]:
# ── Sel bukti rantai sesi (bukti terikat Tahap 4-7 untuk lembar S1) ──────────
# Ringkasan verifikasi yang bisa direkam tanpa mengarang bukti:
#   - /health lokal (device, versi engine)
#   - engine aktif (GET /engines/tts bila tersedia, apa adanya)
#   - rate WAV faktual hanya terbukti dari header generate pertama pada
#     sesi Colab; sel ini TIDAK menggantikan checklist manual S1/G1.
import json
import os
import urllib.request

def get_local(path, headers=None):
    req = urllib.request.Request(f"http://127.0.0.1:3900{path}", headers=headers or {})
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.status, json.load(r)

print("=== BUKTI RANTAI SESI SPIKE (rekam baris ini ke lembar S1) ===")
try:
    s, h = get_local("/health")
    print(f"[health] {s} {h}")
except Exception as e:
    print(f"[health] GAGAL membaca: {e}")

token = os.environ.get("OMNIVOICE_API_KEY", "")
hdr = {"Authorization": f"Bearer {token}"} if token else {}
try:
    s, engines = get_local("/engines/tts", headers=hdr)
    print(f"[engines/tts] {s} {json.dumps(engines, ensure_ascii=False)[:1200]}")
except Exception as e:
    print(f"[engines/tts] tidak terbaca persis ({e}) — bukti engine diambil dari log backend + WAV 48 kHz generate pertama.")

print("[env] OMNIVOICE_TTS_BACKEND =", os.environ.get("OMNIVOICE_TTS_BACKEND"))
print("[data] OMNIVOICE_DATA_DIR =", os.environ.get("OMNIVOICE_DATA_DIR"))

try:
    with open("/content/omnivoice_backend.log", errors="replace") as f:
        logtail = f.read()
    print("[log backend — 3000 karakter terakhir]")
    print(logtail[-3000:])
except OSError as e:
    print(f"[log backend] tidak terbaca: {e}")

print("=== CATATAN UNTUK LEMBAR S1 (isi manual sesudah sesi) ===")
print("- Subclass pemilik id 'voxcpm2' dicatat dari log/WAV generate pertama.")
print("- Rate WAV generate pertama harus 48000 Hz; 24 kHz = fallback tak terbukti = SESI GAGAL, berhenti dan laporkan.")
print("- Perilaku origin 'null' (halaman file://) dicatat hanya dari percobaan nyata.")

---

## Catatan spike S1 (diisi manual pada sesi Colab — jangan diisi dari tebakan)

| Hal yang diverifikasi saat S1 | Hasil faktual (isi saat sesi) |
|---|---|
| Subclass pemilik ID `voxcpm2` (jalur a in-process / jalur b sidecar) | |
| Installer `/models/install` menghormati `HF_HOME`? | |
| Nama field persis `POST /profiles` (`name`, `ref_audio`, `ref_text`, `instruct`, `language`, ...) | |
| Nama field persis `POST /generate` (`text`, `profile_id`/`ref_audio`+`ref_text`, `instruct`, `language`, `seed`, ...) | |
| Perilaku origin `null` (halaman dibuka `file://`) | |
| Rate WAV generate pertama (harus 48000 Hz) + log engine aktif | |
| Device yang dilaporkan `/health` (CUDA/T4 diharapkan) | |
| Respons generate pertama: WAV langsung atau `403 model_licence_required` (pemicu S3) | |

> Isi sel ini hanya dari bukti output sesi nyata. "Kira-kira bisa" bukan keputusan (dokumen 04 §1.1).


---

## Atribusi & lisensi (bundel notebook — JANGAN DIHAPUS)

Notebook ini adalah **fork** dari notebook resmi VoiceStudio
`notebooks/OmniVoice_Studio_Colab.ipynb` (debpalash/VoiceStudio)
yang diunduh pada 5 November 2026. Sel Tahap 1 dan sel Token HF diambil
**persis dari source upstream**; perubahan spike (build UI dilewati,
kurasi model ke VoxCPM2, tunnel cloudflared, Nomor Mesin, env headless)
adalah pekerjaan Stream Host yang diumumkan terbuka di sini.

### VoiceStudio — AGPL-3.0

VoiceStudio dilisensikan di bawah **GNU Affero General Public License
v3 (AGPL-3.0)**. Fork publik ini wajib menyertakan source penuh, teks
lisensi (`LICENSE`), dan atribusi ke upstream. Lihat `LICENSE` dan
`LICENSE-NOTICE.md` di repo fork — keduanya warisan upstream yang
dipertahankan apa adanya, termasuk atribusi penulisnya. Membagikan
notebook ini tanpa `LICENSE`/atribusi adalah pelanggaran AGPL.

Upstream: https://github.com/debpalash/VoiceStudio

### Model — NOTICE

- **VoxCPM2** (`openbmb/VoxCPM2`, kode + bobot) — **Apache License 2.0**.
  NOTICE Apache-2.0 dari OpenBMB wajib dicantumkan di produk dan di repo
  ini (Bagian 8 PRD). Verifikasi metadata: kartu model Hugging Face
  `openbmb/VoxCPM2` menandai `license: apache-2.0` (dicek 5 Nov 2026).
  Catatan jujur: `backend/config/model_licenses.json` pada clone ini
  mencatat record VoxCPM2 sebagai `commercial_use: false` dengan
  `review_status: unreviewed` (inventaris konservatif VoiceStudio) —
  kolom itu **bukan** putusan lisensi upstream; bobot upstream sendiri
  memang Apache-2.0. Inventaris itu **tidak diubah** di fork ini.
- **Whisper** (`Systran/faster-whisper-large-v3`) — untuk transkrip klip
  referensi saja.
- **OmniVoice** (`k2-fsa/OmniVoice`, bobot **CC-BY-NC**) — **DILARANG** untuk
  produk yang dijual. Notebook spike ini **tidak** mengunduhnya dan tidak
  menawarkannya. Bila hasil `generate` pertama bukan WAV 48 kHz (indikasi
  fallback diam-diam), sesi dinyatakan GAGAL dan dilaporkan (bukan ditutupi).

### Peran notebook ini terhadap produk Stream Host

Fork **wajib repo publik** (kewajiban AGPL) — repo ini adalah tempatnya.
Dashboard Stream Host adalah program terpisah (starter MIT) yang hanya
memanggil backend ini lewat HTTP; pemisahan ini dijaga agar kewajiban
AGPL tidak menjalar ke dashboard.
